# Chapter 7: Reading the Planes

## Principal plane extraction from bivectors, plane evolution across layers, and plane similarity

A bivector in $\mathbb{R}^{256}$ has $\binom{256}{2} = 32{,}640$ independent components. That is an enormous object — far too large to inspect by eye. How do we make sense of it?

The answer is **principal plane decomposition**. Every bivector $B$ (a skew-symmetric matrix) can be decomposed into a sum of **simple bivectors**, each representing a single plane of rotation:

$$B = \sigma_1 \, \hat{B}_1 + \sigma_2 \, \hat{B}_2 + \cdots + \sigma_m \, \hat{B}_m$$

where $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_m \geq 0$ are the **rotation magnitudes** (the absolute values of the purely imaginary eigenvalues of $B$), and each $\hat{B}_j$ is a unit simple bivector defining a rotation plane.

This is precisely analogous to PCA for covariance matrices: instead of finding principal *directions*, we find principal *planes*. The top few planes carry most of the rotation, and we can track how these planes evolve across layers to see how the transformer's rotational structure changes with depth.

**By the end of this chapter you will be able to:**
- Extract and rank the principal planes of a bivector
- Measure how many planes are needed to capture most of the rotation
- Track plane evolution across layers
- Compare rotation planes between different layers and different prompts

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga
from layer_time_ga.algebra import Bivector

print("Imports OK")
print(f"NumPy {np.__version__}")

In [ ]:
# Load model and run GA analysis
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")

prompt = "The capital of France is"
result = ltg_ga.analyse(prompt, model=model, compute_dependency=False)

print(f"Model: {model.name}")
print(f"Prompt: \"{prompt}\"")
print(f"Layers: {result.n_layers}, Tokens: {result.n_tokens}, Whitened dim: {result.k}")
print(f"Bivector field: {len(result.bivector_field)} layer transitions")
print(f"Each bivector matrix shape: {result.bivector_field[0].matrix.shape}")
print(f"Independent components per bivector: {result.bivector_field[0].n_components}")

## Principal Planes of a Single Layer

A skew-symmetric matrix $B$ (our bivector) has purely imaginary eigenvalues that come in conjugate pairs $\pm i\sigma_j$, where $\sigma_j \geq 0$. Each pair defines one **principal plane** — a two-dimensional subspace in which the rotation occurs — with rotation magnitude $\sigma_j$.

The eigendecomposition of a $k \times k$ skew-symmetric matrix gives us at most $\lfloor k/2 \rfloor$ such planes. But just as in PCA, the first few planes typically carry the vast majority of the total rotation. The **weight** of each plane is its rotation magnitude $\sigma_j$, and the planes are ordered by decreasing weight.

**Geometric interpretation:** If a layer's bivector has one dominant plane with weight $\sigma_1 \gg \sigma_2$, the layer is performing an "almost simple" rotation — it rotates primarily in a single plane. If multiple planes have comparable weights, the rotation is genuinely higher-dimensional: information is being rearranged across many subspaces simultaneously.

Let us extract the top-5 principal planes from a single layer and visualize their weights.

In [ ]:
# --- Principal planes of layer 20 ---
layer_idx = 20
biv = result.bivector_field[layer_idx]

# Extract top-5 principal planes
planes = biv.principal_planes(n_planes=5)

print(f"=== Principal Planes of Layer Transition {layer_idx} ===")
print(f"Bivector norm: {biv.norm:.4f}")
print(f"Total rotation angle: {biv.angle:.4f} rad ({np.degrees(biv.angle):.1f} deg)")
print()

total_weight = sum(p['weight'] for p in planes)
for i, p in enumerate(planes):
    frac = p['weight'] / total_weight if total_weight > 0 else 0
    print(f"  Plane {i+1}: angle = {p['angle']:.4f} rad, "
          f"weight = {p['weight']:.4f} ({frac:.1%} of total)")

# Bar chart of plane weights
fig, ax = plt.subplots(figsize=(6, 4))
labels = [f"Plane {i+1}" for i in range(len(planes))]
weights = [p['weight'] for p in planes]
colors = ['#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677']
ax.bar(labels, weights, color=colors[:len(planes)], edgecolor='white', linewidth=0.5)
ax.set_ylabel('Rotation magnitude $\\sigma_j$')
ax.set_title(f'Top-5 Principal Plane Weights — Layer {layer_idx}')
ax.axhline(y=0, color='grey', linewidth=0.5)
fig.tight_layout()
plt.show()
print("The dominant plane carries most of the rotation at this layer.")

## How Many Planes Matter?

In principle, a bivector in $\mathbb{R}^k$ can have up to $\lfloor k/2 \rfloor = 128$ nonzero principal planes. In practice, the rotation is concentrated on far fewer. A useful diagnostic is the **cumulative weight fraction**: what fraction of the total rotation magnitude is captured by the top-$n$ planes?

If the top-3 planes carry $> 80\%$ of the total rotation magnitude, the bivector is effectively low-rank — the layer's rotation lives in a 6-dimensional subspace of $\mathbb{R}^{256}$. This is a dramatic dimensionality reduction, and it tells us something fundamental: **the transformer does not use the full rotational freedom available to it**. Its rotations are structured and low-dimensional.

Let us compute this cumulative fraction for every layer.

In [ ]:
# --- Fraction of total rotation in top-1, top-3, top-5 planes per layer ---

n_transitions = len(result.bivector_field)
frac_top1 = np.zeros(n_transitions)
frac_top3 = np.zeros(n_transitions)
frac_top5 = np.zeros(n_transitions)

for l_idx in range(n_transitions):
    biv_l = result.bivector_field[l_idx]

    # Get all eigenvalues of the skew-symmetric matrix
    eigvals = np.linalg.eigvals(biv_l.matrix)
    magnitudes = np.sort(np.abs(eigvals.imag))[::-1]

    # Remove near-duplicate conjugate pairs: keep every other value
    unique_mags = []
    seen = set()
    for m in magnitudes:
        key = round(m, 8)
        if key not in seen and m > 1e-10:
            seen.add(key)
            unique_mags.append(m)
    unique_mags = np.array(unique_mags)

    total = unique_mags.sum() if len(unique_mags) > 0 else 1e-12
    frac_top1[l_idx] = unique_mags[:1].sum() / total if len(unique_mags) >= 1 else 0
    frac_top3[l_idx] = unique_mags[:3].sum() / total if len(unique_mags) >= 1 else 0
    frac_top5[l_idx] = unique_mags[:5].sum() / total if len(unique_mags) >= 1 else 0

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
layers = np.arange(n_transitions)
ax.plot(layers, frac_top1, 'o-', color='#EE6677', linewidth=1.5, markersize=3, label='Top-1 plane')
ax.plot(layers, frac_top3, 's-', color='#4477AA', linewidth=1.5, markersize=3, label='Top-3 planes')
ax.plot(layers, frac_top5, '^-', color='#228833', linewidth=1.5, markersize=3, label='Top-5 planes')
ax.axhline(y=0.8, color='grey', linestyle='--', alpha=0.5, label='80% threshold')
ax.set_xlabel('Layer transition')
ax.set_ylabel('Fraction of total rotation magnitude')
ax.set_title('How Many Principal Planes Carry the Rotation?')
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
plt.show()

print(f"Mean fraction in top-1: {frac_top1.mean():.1%}")
print(f"Mean fraction in top-3: {frac_top3.mean():.1%}")
print(f"Mean fraction in top-5: {frac_top5.mean():.1%}")

## Plane Evolution Across Layers

Now let us track how the **weights** of the top-3 principal planes evolve across layers. This gives us a layer-by-layer view of the rotational structure:

- **Early layers** may concentrate rotation in a single dominant plane (representation formatting).
- **Middle layers** may distribute rotation across several planes (multi-headed information routing).
- **Late layers** may again concentrate rotation as the model narrows toward the prediction.

A stacked area chart makes it easy to see both the total rotation budget and how it is allocated across planes at each layer.

In [ ]:
# --- Stacked area chart: top-3 plane weights across layers ---

n_transitions = len(result.bivector_field)
top3_weights = np.zeros((3, n_transitions))

for l_idx in range(n_transitions):
    planes_l = result.bivector_field[l_idx].principal_planes(n_planes=3)
    for j, p in enumerate(planes_l):
        top3_weights[j, l_idx] = p['weight']

layers = np.arange(n_transitions)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4477AA', '#66CCEE', '#CCBB44']
labels = ['Plane 1 (dominant)', 'Plane 2', 'Plane 3']

ax.stackplot(layers, top3_weights[0], top3_weights[1], top3_weights[2],
             colors=colors, labels=labels, alpha=0.85)
ax.set_xlabel('Layer transition')
ax.set_ylabel('Rotation magnitude $\\sigma_j$')
ax.set_title('Top-3 Principal Plane Weights Across Layers')
ax.legend(loc='upper left')
fig.tight_layout()

save_path = 'ch07_plane_evolution.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()

print("\nThe stacked area shows how the rotation budget is distributed.")
print("Where the colored bands are narrow, rotation is weak overall.")
print("Where Plane 1 dominates, the rotation is nearly simple (rank-2).")

## Plane Similarity Between Layers

Are nearby layers rotating in similar planes, or does every layer choose its own direction? To answer this, we need a measure of **similarity between bivectors**.

Since our bivectors are stored as skew-symmetric matrices $B_l$ and $B_{l'}$, a natural similarity measure is the **normalized Frobenius inner product** (analogous to cosine similarity for vectors):

$$\text{sim}(B_l, B_{l'}) = \frac{\operatorname{tr}(B_l^\top B_{l'})}{\|B_l\|_F \, \|B_{l'}\|_F}$$

This ranges from $-1$ (opposite rotations) through $0$ (orthogonal — completely different planes) to $+1$ (identical rotation planes, possibly different magnitudes).

A heatmap of this similarity matrix reveals:
- **Block structure**: groups of consecutive layers that rotate in similar planes
- **Phase transitions**: sharp drops in similarity marking qualitative changes in processing
- **Long-range correlations**: early and late layers that share rotation planes

In [ ]:
# --- Bivector similarity matrix (all layer pairs) ---

n_transitions = len(result.bivector_field)
sim_matrix = np.zeros((n_transitions, n_transitions))

# Precompute norms
norms = np.array([biv.norm for biv in result.bivector_field])

for i in range(n_transitions):
    Bi = result.bivector_field[i].matrix
    for j in range(n_transitions):
        Bj = result.bivector_field[j].matrix
        inner = np.trace(Bi.T @ Bj)
        denom = norms[i] * norms[j]
        sim_matrix[i, j] = inner / denom if denom > 1e-12 else 0.0

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, cmap='RdBu_r', origin='lower', vmin=-1, vmax=1,
               aspect='equal')
ax.set_xlabel('Layer transition $l\'$')
ax.set_ylabel('Layer transition $l$')
ax.set_title('Bivector Similarity Between Layers\n'
             r'$\mathrm{sim}(B_l, B_{l\'}) = \mathrm{tr}(B_l^\top B_{l\'}) \,/\, \|B_l\| \|B_{l\'}\|$')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Cosine similarity')
fig.tight_layout()

save_path = 'ch07_plane_similarity.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()

# Diagnostic: mean off-diagonal similarity
mask = ~np.eye(n_transitions, dtype=bool)
print(f"\nMean off-diagonal similarity: {sim_matrix[mask].mean():.4f}")
print(f"Mean adjacent-layer similarity: {np.diag(sim_matrix, k=1).mean():.4f}")
print(f"Mean 5-apart similarity: {np.diag(sim_matrix, k=5).mean():.4f}")

## Do Different Prompts Use Different Planes?

A fundamental question: does the transformer rotate in the **same planes** regardless of input, or are the rotation planes **prompt-dependent**?

If planes are universal (input-independent), they reflect the model's architecture — fixed rotational channels baked in by training. If planes are prompt-dependent, the model is dynamically selecting which subspaces to rotate, adapting its geometry to the input.

Let us compare the dominant rotation planes from two different prompts at a late layer (where the model has committed to a prediction strategy).

In [ ]:
# --- Compare rotation planes between two prompts ---

prompt2 = "The derivative of x squared is"
result2 = ltg_ga.analyse(prompt2, model=model, compute_dependency=False)

print(f"Prompt 1: \"{prompt}\"")
print(f"Prompt 2: \"{prompt2}\"")
print(f"Comparing bivectors at a late layer...\n")

# Pick a late layer (use the second-to-last to avoid edge effects)
late_layer = min(len(result.bivector_field), len(result2.bivector_field)) - 2

B1 = result.bivector_field[late_layer].matrix
B2 = result2.bivector_field[late_layer].matrix

# Cosine similarity of the full bivector matrices
inner = np.trace(B1.T @ B2)
norm1 = np.linalg.norm(B1, 'fro')
norm2 = np.linalg.norm(B2, 'fro')
cos_sim = inner / (norm1 * norm2) if (norm1 * norm2) > 1e-12 else 0.0

print(f"=== Bivector Comparison at Layer {late_layer} ===")
print(f"  ||B1||_F = {norm1:.4f}")
print(f"  ||B2||_F = {norm2:.4f}")
print(f"  Cosine similarity: {cos_sim:.4f}")
print()

if abs(cos_sim) > 0.8:
    print("High similarity — these prompts use similar rotation planes at this layer.")
    print("This suggests the rotation planes may be largely architectural (input-independent).")
elif abs(cos_sim) < 0.2:
    print("Low similarity — these prompts use very different rotation planes.")
    print("The model is dynamically selecting which subspaces to rotate.")
else:
    print("Moderate similarity — partial overlap in rotation planes.")
    print("Some planes may be architectural while others are input-dependent.")

# Also compare the dominant plane vectors
planes1 = result.bivector_field[late_layer].principal_planes(n_planes=1)
planes2 = result2.bivector_field[late_layer].principal_planes(n_planes=1)

if planes1 and planes2:
    # Compare via the plane: take the two spanning vectors and compute
    # how aligned the planes are (using the bivector of each plane)
    v1a, v1b = planes1[0]['plane_vectors']
    v2a, v2b = planes2[0]['plane_vectors']
    # Simple bivector of each plane
    P1 = np.outer(v1a, v1b) - np.outer(v1b, v1a)
    P2 = np.outer(v2a, v2b) - np.outer(v2b, v2a)
    plane_sim = np.trace(P1.T @ P2) / (np.linalg.norm(P1, 'fro') * np.linalg.norm(P2, 'fro') + 1e-12)
    print(f"\nDominant plane similarity: {plane_sim:.4f}")

## Exercises

1. **Plane concentration vs. layer depth.** The plot of cumulative weight fractions shows how concentrated the rotation is at each layer. Pick the layer with the *least* concentrated rotation (lowest top-1 fraction) and the *most* concentrated. Extract the top-5 planes for each and compare. What might the difference tell you about what these layers are doing?

2. **Plane similarity decay.** From the similarity heatmap, compute the mean similarity as a function of layer separation $\Delta l = |l - l'|$. Plot $\text{mean sim}(\Delta l)$ vs. $\Delta l$. Does similarity decay smoothly, or are there sharp drops? What would a sharp drop indicate?

3. **Three-prompt comparison.** Run the analysis on three prompts of different types (e.g., factual recall, arithmetic, creative writing). At a fixed late layer, compute the pairwise bivector cosine similarities. Are some prompt pairs more similar than others? Try to explain the pattern.

4. **Plane tracking across layers.** The stacked area chart shows weights but not *identity* of planes. Can two consecutive layers have the same dominant weight but rotate in completely different planes? Devise a method to track whether the dominant plane at layer $l$ is "the same plane" as at layer $l+1$. (Hint: compute the cosine similarity of the rank-1 simple bivectors.)

5. **Rotation rank as a complexity measure.** Define the "rotation rank" of a layer as the number of principal planes needed to capture 90% of the total rotation. Plot this across layers. Does it correlate with any other quantity we have seen (e.g., rotor angle, condition number, holonomy curvature)?